# Fake News Detection with BERT (PyTorch + HuggingFace)

Binary classification of news articles as **real** (1) or **fake** (0).

**Model:** `bert-base-uncased` fine-tuned for sequence classification  
**Dataset:** ISOT Fake News Dataset — ~45 000 labelled articles

> **Training was run offline** via `train_bert.py` and the fine-tuned model is saved to `models/bert_model/`.  
> This notebook loads those weights and focuses on architecture explanation and evaluation.

---

## 1. Background: What is BERT?

### Transformers
BERT is built on the **Transformer** architecture (Vaswani et al., 2017).  Unlike RNNs that process tokens sequentially, Transformers use **self-attention** to relate every token to every other token in parallel:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

This allows the model to capture long-range relationships in a single layer.

---

### BERT: Bidirectional Encoder Representations from Transformers
(Devlin et al., 2019)

BERT is **pre-trained** on two tasks:

| Task | Description |
|------|-------------|
| **Masked Language Modelling (MLM)** | 15 % of tokens are masked; the model predicts them from context |
| **Next Sentence Prediction (NSP)** | Predict whether sentence B follows sentence A |

Pre-training on BookCorpus + English Wikipedia gives BERT rich linguistic knowledge that transfers to downstream tasks via **fine-tuning**.

---

### Fine-tuning for Classification
We attach a single linear layer on top of BERT's `[CLS]` token output:

```
Input tokens  → BERT encoder (12 layers, 768-dim) → [CLS] hidden state → Linear(768, 2) → logits
```

Only a small number of new parameters are added; all BERT weights are updated with a tiny learning rate (~2e-5) to avoid catastrophic forgetting.

---

### WordPiece Tokenisation
BERT uses a **WordPiece** vocabulary (~30 000 subwords).  Rare or unknown words are split into known subword units:

```
"disinformation" → ["dis", "##information"]
```

This eliminates out-of-vocabulary problems and handles morphological variants.

---

### Special Tokens
| Token | Role |
|-------|------|
| `[CLS]` | Prepended to every input; its final hidden state is the sequence representation |
| `[SEP]` | Separates two segments (or marks end of sequence) |
| `[PAD]` | Padding; ignored via the **attention mask** |


## 2. Setup and Imports

Install if needed:
```bash
pip install transformers torch pandas scikit-learn matplotlib seaborn tqdm
```


In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (BertTokenizerFast, BertForSequenceClassification,
                           get_linear_schedule_with_warmup)
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score,
                              classification_report, confusion_matrix)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import random
import warnings
warnings.filterwarnings('ignore')

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')


c:\Users\Tudor\AppData\Local\Programs\Python\Python39\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


## 3. Loading the Dataset & Configuration

We load the dataset solely to **reconstruct the test split** using the same random seed as training.  
Hyperparameters (MAX_LEN, BATCH_SIZE, …) are read from `models/training_results.json` so they exactly match what `train_bert.py` used.

In [ ]:
import json

# Load hyperparams saved during training
with open('models/training_results.json') as f:
    results = json.load(f)

hp         = results['bert']['hyperparams']
MAX_LEN    = hp['max_len']
BATCH_SIZE = hp['batch_size']

print("Hyperparameters used during training:")
for k, v in hp.items():
    print(f"  {k:15s} = {v}")

# Load dataset
fake_df = pd.read_csv('dataset/Fake.csv')
true_df = pd.read_csv('dataset/True.csv')
fake_df['label'] = 0
true_df['label'] = 1

df = pd.concat([fake_df, true_df], ignore_index=True)
df['content'] = df['title'].fillna('') + ' ' + df['text'].fillna('')
df = df[['content', 'label']].dropna().reset_index(drop=True)

# Reproduce the exact same test split used during training (same random_state=42)
_, tmp_texts, _, tmp_labels = train_test_split(
    df['content'].values, df['label'].values,
    test_size=0.2, random_state=42, stratify=df['label'].values
)
_, test_texts, _, test_labels = train_test_split(
    tmp_texts, tmp_labels,
    test_size=0.5, random_state=42, stratify=tmp_labels
)

print(f"\nTest samples : {len(test_texts)}")

## 4. BERT Tokenisation

The `BertTokenizerFast` (backed by the Rust tokenizer library) converts raw text into:

| Output field | Description |
|---|---|
| `input_ids` | Integer IDs for each WordPiece token |
| `attention_mask` | 1 for real tokens, 0 for `[PAD]` |
| `token_type_ids` | Segment IDs (0 for all tokens in single-sentence tasks) |

We set `max_length=256` — long enough to capture most article titles + first paragraphs, short enough to stay within BERT's 512-token limit and fit in memory.


In [ ]:
MODEL_SAVE_PATH = 'models/bert_model'

# Load the tokenizer saved during training (same vocabulary as fine-tuning)
tokenizer = BertTokenizerFast.from_pretrained(MODEL_SAVE_PATH)

# Illustrate tokenisation on a sample
sample = test_texts[0][:200]
enc = tokenizer(sample, max_length=MAX_LEN, truncation=True, padding='max_length',
                return_tensors='pt')

print("Sample text (first 200 chars):")
print(sample)
print(f"\nTokens  : {tokenizer.convert_ids_to_tokens(enc['input_ids'][0])[:20]} ...")
print(f"input_ids shape    : {enc['input_ids'].shape}")
print(f"attention_mask sum : {enc['attention_mask'].sum().item()}  (real tokens)")

## 5. PyTorch Dataset and DataLoader

Each sample from the Dataset returns a dictionary with `input_ids`, `attention_mask`, and `label`.  The DataLoader collates these into mini-batches automatically.


In [ ]:
class FakeNewsBertDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN):
        self.encodings = tokenizer(
            list(texts),
            max_length=max_len,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'label':          self.labels[idx]
        }

# Only the test split is needed for evaluation
print("Encoding test set...")
test_ds     = FakeNewsBertDataset(test_texts, test_labels, tokenizer)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"Test samples  : {len(test_ds)}")
print(f"Test batches  : {len(test_loader)}")

## 6. Loading the Saved Model

We load the fine-tuned weights directly from `models/bert_model/` using HuggingFace's `from_pretrained()`.  
This is the model that was trained and saved by `train_bert.py`.

```
models/bert_model/
  ├── config.json            ← architecture configuration
  ├── pytorch_model.bin      ← fine-tuned weights
  ├── vocab.txt              ← WordPiece vocabulary
  └── tokenizer_config.json  ← tokenizer settings
```

In [ ]:
model = BertForSequenceClassification.from_pretrained(MODEL_SAVE_PATH)
model = model.to(device)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded from   : {MODEL_SAVE_PATH}")
print(f"Total parameters    : {total_params:,}")
print(f"\nRecorded test accuracy : {results['bert']['test_acc']:.4f}")
print(f"Recorded test F1       : {results['bert']['test_f1']:.4f}")

## 7. Training (Reference)

Training was run offline via:
```bash
python train_bert.py          # 12 000 samples, 3 epochs  (GPU recommended)
python train_bert.py --quick  # 2 000 samples,  2 epochs  (CPU-friendly)
```

### What happens inside the script

| Component | Choice | Reason |
|-----------|--------|--------|
| **Model** | `bert-base-uncased` | Standard English BERT, widely benchmarked |
| **Optimiser** | AdamW (lr = 2×10⁻⁵, weight decay = 0.01) | Decoupled weight decay prevents over-regularising biases |
| **LR schedule** | Linear warm-up (10 %) + linear decay | Avoids catastrophic forgetting at the start of fine-tuning |
| **Gradient clipping** | max norm = 1.0 | Stabilises training, same as the original BERT paper |
| **Epochs** | 3 (full) / 2 (quick) | BERT typically converges fast on classification tasks |
| **Checkpointing** | Best validation accuracy | Restores the generalising checkpoint before test evaluation |

### Saved artefacts
| File | Contents |
|------|----------|
| `models/bert_model/pytorch_model.bin` | Fine-tuned weights |
| `models/bert_model/config.json` | Architecture configuration |
| `models/bert_model/vocab.txt` + tokenizer files | Tokenizer (required for inference) |
| `models/training_results.json` → `"bert"` key | Hyperparams + per-epoch history + test metrics |

In [ ]:
def bert_eval_epoch(model, loader):
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)
            outputs = model(input_ids=input_ids,
                            attention_mask=attention_mask,
                            labels=labels)
            total_loss += outputs.loss.item()
            all_preds.extend(outputs.logits.argmax(1).cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
    return total_loss / len(loader), accuracy_score(all_labels, all_preds), all_preds, all_labels

## 9. Evaluation on the Test Set


In [ ]:
_, test_acc_run, test_preds_run, test_labels_run = bert_eval_epoch(model, test_loader)

print(f"Test Accuracy  : {test_acc_run:.4f}")
print(f"Test F1 (macro): {f1_score(test_labels_run, test_preds_run, average='macro'):.4f}")
print()
print(classification_report(test_labels_run, test_preds_run, target_names=['Fake', 'Real']))

In [ ]:
# Load training history from the saved results JSON
history = results['bert']['history']

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
epochs = range(1, len(history['train_loss']) + 1)

# Loss curve
axes[0].plot(epochs, history['train_loss'], 'o-', label='Train', linewidth=2)
axes[0].plot(epochs, history['val_loss'],   's--', label='Val',  linewidth=2)
axes[0].set_title('BERT — Loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Accuracy curve
axes[1].plot(epochs, history['train_acc'], 'o-', label='Train', linewidth=2)
axes[1].plot(epochs, history['val_acc'],   's--', label='Val',  linewidth=2)
axes[1].set_title('BERT — Accuracy')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0, 1.15); axes[1].set_yticks([0, 0.25, 0.5, 0.75, 1.0])
axes[1].legend(); axes[1].grid(True, alpha=0.3)

# Confusion matrix
cm = confusion_matrix(test_labels_run, test_preds_run)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', ax=axes[2],
            xticklabels=['Fake', 'Real'], yticklabels=['Fake', 'Real'])
axes[2].set_title('BERT — Confusion Matrix')
axes[2].set_ylabel('True'); axes[2].set_xlabel('Predicted')

plt.suptitle('BERT Fine-tuning Results', fontsize=14)
plt.tight_layout()
plt.show()

## 10. Conclusion

### Why BERT outperforms RNN / LSTM

| Aspect | RNN / LSTM | BERT |
|--------|-----------|------|
| **Pre-trained knowledge** | None (random init) | 110M params pre-trained on ~3 B words |
| **Context window** | Sequential; degrades on long texts | Full 512-token self-attention |
| **Subword vocab** | Fixed word vocab; OOV problem | WordPiece; handles any word |
| **Training time** | Minutes | ~Hours (fine-tuning) |
| **Parameters** | ~4 M | ~110 M |

### When to choose what
- **RNN / LSTM**: resource-constrained environments, very low latency inference, or interpretability requirements.
- **BERT**: when accuracy is paramount and you have access to a GPU for fine-tuning.

### Further improvements
- **`roberta-base`** or **`distilbert-base-uncased`** (faster, often comparable accuracy).
- **Data augmentation** (back-translation, paraphrase) to improve recall on edge cases.
- **Attention visualisation** (`BertViz`) to inspect which tokens the model attends to.
